# Date And Time Helper Agent

In this notebook, I build a small agent that answers questions about dates, using the Hugging Face `smolagents` library.

Like the earlier calculator/converter and text stats/formatter agents, this one does not guess. It works out date differences and reformats dates by calling tools built on Python's own `datetime` module, instead of asking the language model to do date arithmetic in its head.

In this notebook, I will learn how to:

- Write a plain Python function that counts the days between two dates
- Write a plain Python function that reformats a date into a different style
- Turn each one into an agent tool with the `@tool` decorator
- Reject an invalid date or an unknown style inside a tool instead of guessing
- Give an agent both tools and watch it choose, or chain, the right one

Everything here stays small on purpose, so the whole idea fits in one sitting.

## 1. Importing Libraries and Creating the Model

First I import the pieces I need from `smolagents`, plus `datetime` from the standard library for the actual date arithmetic.

`CodeAgent` is the agent that writes and runs Python code to solve a task. `tool` is the decorator I use to turn a plain function into something an agent can call. `InferenceClientModel` is the language model, which runs on the Hugging Face Inference API rather than on my own machine.

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, tool
from datetime import datetime

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)

## 2. Writing a Basic Date Difference Function

Before I build a tool, I write the date arithmetic as an ordinary Python function.

It takes two dates as plain strings in `YYYY-MM-DD` format and returns the number of days between them. Keeping the logic in a plain function first means I can test it on its own, without an agent or a language model anywhere near it.

In [ ]:
def days_between(start: str, end: str) -> int:
    """Returns the number of days between two ISO dates."""
    start_date = datetime.strptime(start, "%Y-%m-%d")
    end_date = datetime.strptime(end, "%Y-%m-%d")
    return (end_date - start_date).days

## 3. Testing the Date Difference Function

I try the function on a few dates I can check by hand before trusting it with anything else.

In [ ]:
print(days_between("2026-01-01", "2026-01-31"))
print(days_between("2026-03-01", "2026-03-01"))
print(days_between("2026-06-15", "2026-01-01"))